# Curve fitting - Artificial Neural Networks

In this activity, we will use a simple artificial neural network to fit a model the provides the best decision boundary to the data provided. At this point of the lecture, you will learn that while traditional curve fitting might struggle with non-linear relationships or interactions between multiple variables, neural networks excel in these areas due to their layered architecture and non-linear activation functions. Essentially, a neural network can be thought of as performing a highly sophisticated form of curve fitting. It adjusts its internal parameters (weights and biases) during training to minimize the loss function, analogous to how coefficients are adjusted in polynomial curve fitting. This enables neural networks to fit intricate patterns in data, making them powerful tools for tasks that involve image recognition, natural language processing, and more, where traditional models might fail to capture the complexity of the data.

This notebook assists learning the advantages of combinining non-linear activations with a more complex model architecture i.e., a neural network.

## Neural network architecture

The following illustration is our model architecture. It is a simple neural network with two inputs (i.e., `Feature 0` and `Feature 1`), a single `hidden` layer with two `nodes` or `units`, and an output layer with a single `node`. In binary classification tasks, meaning, those that answer "Yes"/"No", "Positive"/"Negative", and "Class 0" vs "Class 1" questions, a single output node will do. The values here are then thresholded, meaning we set a value decision boundary and anything north or south of that value are mapped to the two classes of outputs. The labels in the illustration should guide you when changing the slider values for each parameter in the network.


<center><img src="https://github.com/mikedataCrunch/GMS5204/blob/main/media/nn_activity.jpeg?raw=true"/></center>
<center><b>Figure 1. Simple neural network architecture: 2 input features, 1 hidden layer with 2 nodes, and an output layer with a single node.</b></center>

## Data description
The sample data we're using here resembles a binary classification, where each sample belongs to either `Class: 0` or `Class: 1`. This is quite common in tasks that requires identifying examples that are `positive` to a particular condition, disease, diagnosis, or some other classification criteria.

The binary classification task takes in two input `features`. We can think of features as characteristics of an example. If consider humans as examples, then the two features can be height & weight, age & gender, gender & income, etc. If we consider this in a medical sense, then the two features can be test result A & B, lifestyle & age, or whatever pair of characteristics available to us.

In our activity, we will refer to these as `Feature 0` and `Feature 1`.

In [14]:
import json
import numpy as np
from IPython.display import HTML, display

def ReLU(x):
    """ReLU: Rectified linear unit function."""
    return np.maximum(0, x)

# Define the neural network with one hidden layer
def simple_neural_network(inputs, w1, w2, b1, b2):
    # Inputs is expected to be Nx2, w1 is 2x2, b1 is size 2, w2 is size 2, b2 is a scalar
    hidden_layer_input = np.dot(inputs, w1) + b1
    hidden_layer_activation = ReLU(hidden_layer_input) 
    output = np.dot(hidden_layer_activation, w2) + b2
    return 1 / (1 + np.exp(-output))  # Sigmoid activation for output layer or output activation

def calculate_bce_loss(y_true, y_pred):
    """
    Calculate the binary cross-entropy loss.

    Parameters:
    -----------
    y_true (array-like): True binary labels (0 or 1).
    y_pred (array-like): Predicted probabilities, between 0 and 1.

    Returns:
    --------
    float: The average binary cross-entropy loss.
    """
    # Ensure that y_pred does not contain values exactly equal to 0 or 1,
    # as log(0) is undefined and can cause computation errors.
    epsilon = 1e-10
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    # Calculate binary cross-entropy loss
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return loss

In [15]:

# Generate a simple dataset
np.random.seed(42)
# Class 0
feature_0_class_0 = np.random.normal(2, 1, 100)  # Feature 1 for class 0
feature_1_class_0 = np.random.normal(2, 1, 100)  # Feature 2 for class 0
# Class 1
feature_0_class_1 = np.random.normal(5, 1, 100)  # Feature 1 for class 1
feature_1_class_1 = np.random.normal(5, 1, 100)  # Feature 2 for class 1

features = np.vstack((np.column_stack((feature_0_class_0, feature_1_class_0)),
                      np.column_stack((feature_0_class_1, feature_1_class_1))))
y_true = np.array([0]*100 + [1]*100)

In [17]:
# Embed the data in a browser-side neural-network activity.
nn_widget = """
<div id="nn-curve-fitting-widget">
  <style>
    #nn-curve-fitting-widget {
      max-width: 950px; padding: 16px; border: 1px solid #d9d9d9;
      border-radius: 10px; font-family: Arial, sans-serif; background: white;
    }
    #nn-curve-fitting-widget .architecture {
      padding: 10px; margin-bottom: 12px; border-radius: 8px; text-align: center;
      background: #f3f4f6; color: #1f2937; font-weight: 600;
    }
    #nn-curve-fitting-widget .control-groups { display: grid; grid-template-columns: repeat(auto-fit, minmax(275px, 1fr)); gap: 12px; }
    #nn-curve-fitting-widget fieldset { margin: 0; padding: 10px; border: 1px solid #d1d5db; border-radius: 8px; }
    #nn-curve-fitting-widget legend { padding: 0 6px; font-weight: 600; color: #374151; }
    #nn-curve-fitting-widget label { display: grid; grid-template-columns: 92px 1fr 55px; gap: 8px; align-items: center; margin: 7px 0; }
    #nn-curve-fitting-widget input[type=range] { width: 100%; }
    #nn-curve-fitting-widget output { font-variant-numeric: tabular-nums; text-align: right; }
    #nn-curve-fitting-widget .view-controls { display: flex; flex-wrap: wrap; align-items: center; gap: 8px; margin-top: 12px; }
    #nn-curve-fitting-widget .view-controls button { padding: 7px 12px; border: 1px solid #9ca3af; border-radius: 6px; background: #f9fafb; cursor: pointer; }
    #nn-curve-fitting-widget .view-controls button:hover { background: #eef2ff; }
    #nn-curve-fitting-widget .view-controls span { margin-left: 4px; color: #374151; font-weight: 600; }
    #nn-curve-fitting-widget .metric { margin: 14px 0 4px; font-size: 17px; font-weight: 600; }
    #nn-curve-fitting-widget canvas { width: 100%; height: auto; display: block; }
  </style>
  <div class="architecture">2 inputs &rarr; 2 ReLU hidden units &rarr; 1 sigmoid output</div>
  <div class="control-groups">
    <fieldset><legend>Input &rarr; hidden weights</legend>
      <label>x0 &rarr; h0 <input data-role="w1-00" type="range" min="-10" max="10" step="0.01" value="2"><output data-role="w1-00-value">2.00</output></label>
      <label>x0 &rarr; h1 <input data-role="w1-01" type="range" min="-10" max="10" step="0.01" value="2"><output data-role="w1-01-value">2.00</output></label>
      <label>x1 &rarr; h0 <input data-role="w1-10" type="range" min="-10" max="10" step="0.01" value="2"><output data-role="w1-10-value">2.00</output></label>
      <label>x1 &rarr; h1 <input data-role="w1-11" type="range" min="-10" max="10" step="0.01" value="2"><output data-role="w1-11-value">2.00</output></label>
    </fieldset>
    <fieldset><legend>Hidden layer biases</legend>
      <label>bias h0 <input data-role="b1-0" type="range" min="-10" max="10" step="0.01" value="-1"><output data-role="b1-0-value">-1.00</output></label>
      <label>bias h1 <input data-role="b1-1" type="range" min="-10" max="10" step="0.01" value="-1"><output data-role="b1-1-value">-1.00</output></label>
    </fieldset>
    <fieldset><legend>Hidden &rarr; output</legend>
      <label>h0 &rarr; out <input data-role="w2-0" type="range" min="-10" max="10" step="0.01" value="0.5"><output data-role="w2-0-value">0.50</output></label>
      <label>h1 &rarr; out <input data-role="w2-1" type="range" min="-10" max="10" step="0.01" value="-0.4"><output data-role="w2-1-value">-0.40</output></label>
      <label>output bias <input data-role="b2" type="range" min="-10" max="10" step="0.01" value="-1"><output data-role="b2-value">-1.00</output></label>
    </fieldset>
  </div>
  <div class="view-controls" aria-label="Plot zoom controls">
    <button data-role="zoom-out" type="button">&minus; Zoom out</button>
    <button data-role="zoom-reset" type="button">Reset view</button>
    <button data-role="zoom-in" type="button">&plus; Zoom in</button>
    <span>View: <output data-role="zoom-value">1.00&times;</output></span>
  </div>
  <div class="metric">BCE loss: <span data-role="loss"></span> <small>(lower is better)</small> &nbsp;|&nbsp; Accuracy: <span data-role="accuracy"></span></div>
  <canvas data-role="plot" width="950" height="620" role="img" aria-label="Neural-network probabilities and nonlinear decision boundary"></canvas>
</div>
<script>
(() => {
  const root = document.getElementById('nn-curve-fitting-widget');
  const canvas = root.querySelector('[data-role=plot]');
  const context = canvas.getContext('2d');
  const features = __FEATURES__;
  const labels = __LABELS__;
  const roles = ['w1-00', 'w1-01', 'w1-10', 'w1-11', 'b1-0', 'b1-1', 'w2-0', 'w2-1', 'b2'];
  const inputs = Object.fromEntries(roles.map(role => [role, root.querySelector(`[data-role=${role}]`)]));
  const area = {left: 70, right: 850, top: 25, bottom: 550};
  const padding = 0.5;
  const baseView = {
    xMin: Math.min(...features.map(point => point[0])) - padding,
    xMax: Math.max(...features.map(point => point[0])) + padding,
    yMin: Math.min(...features.map(point => point[1])) - padding,
    yMax: Math.max(...features.map(point => point[1])) + padding,
  };
  const viewCenter = {x: (baseView.xMin + baseView.xMax) / 2, y: (baseView.yMin + baseView.yMax) / 2};
  let zoomLevel = 1;
  let view = {...baseView};
  const scaleX = value => area.left + (value - view.xMin) / (view.xMax - view.xMin) * (area.right - area.left);
  const scaleY = value => area.bottom - (value - view.yMin) / (view.yMax - view.yMin) * (area.bottom - area.top);
  const unscaleX = pixel => view.xMin + (pixel - area.left) / (area.right - area.left) * (view.xMax - view.xMin);
  const unscaleY = pixel => view.yMax - (pixel - area.top) / (area.bottom - area.top) * (view.yMax - view.yMin);
  const sigmoid = value => value >= 0 ? 1 / (1 + Math.exp(-value)) : Math.exp(value) / (1 + Math.exp(value));

  function setZoom(nextZoom) {
    zoomLevel = Math.max(0.25, Math.min(4, nextZoom));
    const halfWidth = (baseView.xMax - baseView.xMin) / (2 * zoomLevel);
    const halfHeight = (baseView.yMax - baseView.yMin) / (2 * zoomLevel);
    view = {
      xMin: viewCenter.x - halfWidth, xMax: viewCenter.x + halfWidth,
      yMin: viewCenter.y - halfHeight, yMax: viewCenter.y + halfHeight,
    };
    root.querySelector('[data-role=zoom-value]').textContent = `${zoomLevel.toFixed(2)}×`;
  }

  function parameters() {
    return Object.fromEntries(roles.map(role => [role, Number(inputs[role].value)]));
  }

  // This exactly matches the notebook architecture: 2 inputs, 2 ReLU units, 1 sigmoid output.
  function networkProbability(x0, x1, values) {
    const hidden0 = Math.max(0, x0 * values['w1-00'] + x1 * values['w1-10'] + values['b1-0']);
    const hidden1 = Math.max(0, x0 * values['w1-01'] + x1 * values['w1-11'] + values['b1-1']);
    return sigmoid(hidden0 * values['w2-0'] + hidden1 * values['w2-1'] + values.b2);
  }

  function probabilityColor(probability) {
    const blue = [59, 76, 192];
    const white = [245, 245, 245];
    const red = [180, 4, 38];
    const start = probability < 0.5 ? blue : white;
    const end = probability < 0.5 ? white : red;
    const amount = probability < 0.5 ? probability * 2 : (probability - 0.5) * 2;
    const channel = index => Math.round((start[index] + (end[index] - start[index]) * amount) * 0.45 + 255 * 0.55);
    return `rgb(${channel(0)}, ${channel(1)}, ${channel(2)})`;
  }

  function drawAxes() {
    context.strokeStyle = '#d1d5db'; context.lineWidth = 1;
    context.fillStyle = '#555'; context.font = '12px Arial'; context.textAlign = 'center';
    for (let i = 0; i <= 5; i++) {
      const px = area.left + i * (area.right - area.left) / 5;
      const py = area.top + i * (area.bottom - area.top) / 5;
      context.beginPath(); context.moveTo(px, area.top); context.lineTo(px, area.bottom); context.stroke();
      context.beginPath(); context.moveTo(area.left, py); context.lineTo(area.right, py); context.stroke();
      context.fillText((view.xMin + i * (view.xMax - view.xMin) / 5).toFixed(1), px, area.bottom + 20);
      context.textAlign = 'right'; context.fillText((view.yMax - i * (view.yMax - view.yMin) / 5).toFixed(1), area.left - 9, py + 4); context.textAlign = 'center';
    }
    context.strokeStyle = '#333'; context.lineWidth = 1.5; context.strokeRect(area.left, area.top, area.right - area.left, area.bottom - area.top);
    const gradient = context.createLinearGradient(0, area.bottom, 0, area.top);
    gradient.addColorStop(0, probabilityColor(0)); gradient.addColorStop(0.5, probabilityColor(0.5)); gradient.addColorStop(1, probabilityColor(1));
    context.fillStyle = gradient; context.fillRect(870, area.top, 18, area.bottom - area.top);
    context.strokeStyle = '#555'; context.lineWidth = 1; context.strokeRect(870, area.top, 18, area.bottom - area.top);
    context.fillStyle = '#555'; context.font = '12px Arial'; context.textAlign = 'left';
    context.fillText('1.0', 893, area.top + 4); context.fillText('0.5', 893, (area.top + area.bottom) / 2 + 4); context.fillText('0.0', 893, area.bottom + 4);
    context.save(); context.translate(935, (area.top + area.bottom) / 2); context.rotate(-Math.PI / 2); context.textAlign = 'center'; context.fillText('Predicted probability of class 1', 0, 0); context.restore();
    context.fillStyle = '#333'; context.font = '15px Arial'; context.textAlign = 'center';
    context.fillText('Feature 0', (area.left + area.right) / 2, 608);
    context.save(); context.translate(18, (area.top + area.bottom) / 2); context.rotate(-Math.PI / 2); context.fillText('Feature 1', 0, 0); context.restore();
  }

  function edgeCrossing(pointA, valueA, pointB, valueB) {
    const crosses = (valueA < 0.5 && valueB >= 0.5) || (valueA >= 0.5 && valueB < 0.5);
    if (!crosses) return null;
    const amount = (0.5 - valueA) / (valueB - valueA);
    return [pointA[0] + amount * (pointB[0] - pointA[0]), pointA[1] + amount * (pointB[1] - pointA[1])];
  }

  function drawBoundary(probabilities, columns, rows) {
    context.save(); context.strokeStyle = '#111'; context.lineWidth = 2.4; context.setLineDash([8, 6]);
    let segmentCount = 0;
    for (let column = 0; column < columns; column++) {
      for (let row = 0; row < rows; row++) {
        const x0 = area.left + column * (area.right - area.left) / columns;
        const x1 = area.left + (column + 1) * (area.right - area.left) / columns;
        const y0 = area.top + row * (area.bottom - area.top) / rows;
        const y1 = area.top + (row + 1) * (area.bottom - area.top) / rows;
        const topLeft = probabilities[row][column];
        const topRight = probabilities[row][column + 1];
        const bottomRight = probabilities[row + 1][column + 1];
        const bottomLeft = probabilities[row + 1][column];
        const crossings = [
          edgeCrossing([x0, y0], topLeft, [x1, y0], topRight),
          edgeCrossing([x1, y0], topRight, [x1, y1], bottomRight),
          edgeCrossing([x1, y1], bottomRight, [x0, y1], bottomLeft),
          edgeCrossing([x0, y1], bottomLeft, [x0, y0], topLeft),
        ].filter(Boolean);
        for (let index = 0; index + 1 < crossings.length; index += 2) {
          context.beginPath(); context.moveTo(...crossings[index]); context.lineTo(...crossings[index + 1]); context.stroke(); segmentCount++;
        }
      }
    }
    context.restore();
    return segmentCount;
  }

  function update() {
    const values = parameters();
    context.clearRect(0, 0, canvas.width, canvas.height);
    const columns = 100;
    const rows = 70;
    const cellWidth = (area.right - area.left) / columns;
    const cellHeight = (area.bottom - area.top) / rows;
    const probabilities = Array.from({length: rows + 1}, (_, row) =>
      Array.from({length: columns + 1}, (_, column) => networkProbability(
        unscaleX(area.left + column * cellWidth), unscaleY(area.top + row * cellHeight), values
      ))
    );
    for (let column = 0; column < columns; column++) {
      for (let row = 0; row < rows; row++) {
        const probability = (probabilities[row][column] + probabilities[row][column + 1] + probabilities[row + 1][column] + probabilities[row + 1][column + 1]) / 4;
        context.fillStyle = probabilityColor(probability);
        context.fillRect(area.left + column * cellWidth, area.top + row * cellHeight, cellWidth + 1, cellHeight + 1);
      }
    }
    drawAxes();
    const segmentCount = drawBoundary(probabilities, columns, rows);
    if (segmentCount === 0) {
      context.fillStyle = 'rgba(255,255,255,0.88)'; context.fillRect(area.left + 10, area.bottom - 32, 330, 24);
      context.fillStyle = '#444'; context.font = '12px Arial'; context.textAlign = 'left';
      context.fillText('No 50% decision boundary is visible with these settings.', area.left + 17, area.bottom - 16);
    }
    context.save();
    context.beginPath(); context.rect(area.left, area.top, area.right - area.left, area.bottom - area.top); context.clip();
    for (let i = 0; i < features.length; i++) {
      context.beginPath(); context.arc(scaleX(features[i][0]), scaleY(features[i][1]), 4.5, 0, 2 * Math.PI);
      context.fillStyle = labels[i] === 0 ? '#4169e1' : '#dc143c'; context.fill();
      context.strokeStyle = 'rgba(255,255,255,0.8)'; context.lineWidth = 0.8; context.stroke();
    }
    context.restore();
    context.fillStyle = '#4169e1'; context.fillRect(685, 570, 14, 14);
    context.fillStyle = '#333'; context.font = '13px Arial'; context.textAlign = 'left'; context.fillText('Class 0', 705, 582);
    context.fillStyle = '#dc143c'; context.fillRect(780, 570, 14, 14);
    context.fillStyle = '#333'; context.fillText('Class 1', 800, 582);

    let loss = 0;
    let correct = 0;
    for (let i = 0; i < features.length; i++) {
      const probability = networkProbability(features[i][0], features[i][1], values);
      const clipped = Math.max(1e-10, Math.min(1 - 1e-10, probability));
      loss -= labels[i] * Math.log(clipped) + (1 - labels[i]) * Math.log(1 - clipped);
      if ((probability >= 0.5 ? 1 : 0) === labels[i]) correct++;
    }
    for (const role of roles) root.querySelector(`[data-role=${role}-value]`).textContent = values[role].toFixed(2);
    root.querySelector('[data-role=loss]').textContent = (loss / features.length).toFixed(4);
    root.querySelector('[data-role=accuracy]').textContent = `${(correct / features.length * 100).toFixed(1)}%`;
  }

  for (const input of Object.values(inputs)) input.addEventListener('input', update);
  root.querySelector('[data-role=zoom-out]').addEventListener('click', () => { setZoom(zoomLevel / 1.25); update(); });
  root.querySelector('[data-role=zoom-reset]').addEventListener('click', () => { setZoom(1); update(); });
  root.querySelector('[data-role=zoom-in]').addEventListener('click', () => { setZoom(zoomLevel * 1.25); update(); });
  setZoom(1);
  update();
})();
</script>
"""
nn_widget = nn_widget.replace('__FEATURES__', json.dumps(features.tolist()))
nn_widget = nn_widget.replace('__LABELS__', json.dumps(y_true.tolist()))
display(HTML(nn_widget))


## A harder 2-D problem: interleaving spirals

The two classes below wind around one another, so a straight line—and even a very small network—cannot describe the decision boundary well. A useful starting architecture is two `tanh` hidden layers. The first layer learns local directions and bends; the second combines them into the repeated curved regions needed to separate the spirals.

```mermaid
flowchart LR
    I["2 inputs<br/>Feature 0, Feature 1"] --> H1["Hidden layer 1<br/>16 tanh units"]
    H1 --> H2["Hidden layer 2<br/>16 tanh units"]
    H2 --> O["1 sigmoid output<br/>P(Class 1)"]
```

Use the knobs to change both the problem and the model. **Noise**, **turns**, and **samples per class** control the dataset; **hidden units**, **learning rate**, and **training epochs** control learning; and the **decision threshold** controls how probabilities become class predictions. Changing a data or architecture knob resets the model.

In [ ]:
spiral_widget = r"""
<div id="spiral-nn-widget">
<style>
#spiral-nn-widget{max-width:950px;padding:16px;border:1px solid #d9d9d9;border-radius:10px;font-family:Arial,sans-serif;background:#fff;color:#1f2937}
#spiral-nn-widget .arch{padding:10px;margin-bottom:12px;border-radius:8px;text-align:center;background:#f3f4f6;font-weight:600}
#spiral-nn-widget .controls{display:grid;grid-template-columns:repeat(auto-fit,minmax(270px,1fr));gap:12px}
#spiral-nn-widget fieldset{margin:0;padding:10px;border:1px solid #d1d5db;border-radius:8px}
#spiral-nn-widget legend{padding:0 6px;font-weight:600}
#spiral-nn-widget label{display:grid;grid-template-columns:120px 1fr 55px;gap:8px;align-items:center;margin:8px 0}
#spiral-nn-widget input[type=range]{width:100%}
#spiral-nn-widget output{font-variant-numeric:tabular-nums;text-align:right}
#spiral-nn-widget .actions{display:flex;flex-wrap:wrap;align-items:center;gap:9px;margin:13px 0}
#spiral-nn-widget button{padding:8px 13px;border:1px solid #9ca3af;border-radius:6px;background:#f9fafb;cursor:pointer}
#spiral-nn-widget button.primary{background:#4f46e5;color:#fff;border-color:#4338ca}
#spiral-nn-widget button:disabled{opacity:.55;cursor:wait}
#spiral-nn-widget .status{font-weight:600;margin-left:4px}
#spiral-nn-widget .metrics{margin:8px 0 4px;font-size:16px;font-weight:600}
#spiral-nn-widget .hint{font-size:13px;color:#4b5563;margin:4px 0 10px}
#spiral-nn-widget canvas{width:100%;height:auto;display:block}
</style>
<div class="arch" data-role="architecture"></div>
<div class="controls">
 <fieldset><legend>Dataset knobs</legend>
  <label>Samples / class <input data-role="samples" type="range" min="40" max="250" step="10" value="120"><output></output></label>
  <label>Noise <input data-role="noise" type="range" min="0" max="0.20" step="0.01" value="0.04"><output></output></label>
  <label>Spiral turns <input data-role="turns" type="range" min="1" max="3" step="0.1" value="2"><output></output></label>
  <label>Random seed <input data-role="seed" type="range" min="1" max="99" step="1" value="7"><output></output></label>
 </fieldset>
 <fieldset><legend>Network and training knobs</legend>
  <label>Hidden units <input data-role="width" type="range" min="4" max="32" step="2" value="16"><output></output></label>
  <label>Learning rate <input data-role="rate" type="range" min="0.001" max="0.05" step="0.001" value="0.012"><output></output></label>
  <label>Training epochs <input data-role="epochs" type="range" min="50" max="800" step="50" value="300"><output></output></label>
  <label>Decision threshold <input data-role="threshold" type="range" min="0.10" max="0.90" step="0.01" value="0.50"><output></output></label>
 </fieldset>
</div>
<div class="actions"><button class="primary" data-role="train" type="button">Train network</button><button data-role="reset" type="button">Reset model</button><button data-role="new-data" type="button">New data</button><span class="status" data-role="status"></span></div>
<div class="metrics">BCE loss: <span data-role="loss">—</span> &nbsp;|&nbsp; Accuracy: <span data-role="accuracy">—</span></div>
<div class="hint">The shaded surface is the predicted probability of Class 1; the dashed curve is the selected decision threshold.</div>
<canvas data-role="plot" width="950" height="620" role="img" aria-label="Two spiral classes and a neural-network decision boundary"></canvas>
</div>
<script>
(() => {
 const root=document.getElementById('spiral-nn-widget'), canvas=root.querySelector('[data-role=plot]'), ctx=canvas.getContext('2d');
 const names=['samples','noise','turns','seed','width','rate','epochs','threshold'];
 const knobs=Object.fromEntries(names.map(n=>[n,root.querySelector(`[data-role=${n}]`)]));
 const area={left:65,right:865,top:22,bottom:550}; let data=[],model=null,runToken=0;
 function value(n){return Number(knobs[n].value)}
 function showKnobs(){for(const n of names){const v=value(n);knobs[n].nextElementSibling.textContent=['noise','rate','threshold'].includes(n)?v.toFixed(3):n==='turns'?v.toFixed(1):v.toFixed(0)} root.querySelector('[data-role=architecture]').textContent=`2 inputs → ${value('width')} tanh → ${value('width')} tanh → 1 sigmoid output`;}
 function rng(seed){let s=seed>>>0;return()=>{s=(1664525*s+1013904223)>>>0;return s/4294967296}}
 function normal(random){const u=Math.max(random(),1e-9),v=random();return Math.sqrt(-2*Math.log(u))*Math.cos(2*Math.PI*v)}
 function makeData(){const random=rng(value('seed')),n=value('samples'),noise=value('noise'),turns=value('turns');data=[];for(let c=0;c<2;c++)for(let i=0;i<n;i++){const u=(i+.5)/n,r=.08+.92*u,a=turns*2*Math.PI*u+c*Math.PI+normal(random)*noise*2.2;data.push({x:r*Math.cos(a)+normal(random)*noise,y:r*Math.sin(a)+normal(random)*noise,label:c})} resetModel();}
 function makeArray(n,fn){return Array.from({length:n},(_,i)=>fn(i))}
 function resetModel(){runToken++;const h=value('width'),random=rng(value('seed')+104729),rand=scale=>(random()*2-1)*scale;model={h,t:0,W1:makeArray(2*h,()=>rand(Math.sqrt(6/(2+h)))),b1:makeArray(h,()=>0),W2:makeArray(h*h,()=>rand(Math.sqrt(6/(2*h)))),b2:makeArray(h,()=>0),W3:makeArray(h,()=>rand(Math.sqrt(6/(h+1)))),b3:[0]};for(const key of ['W1','b1','W2','b2','W3','b3']){model['m'+key]=model[key].map(()=>0);model['v'+key]=model[key].map(()=>0)}root.querySelector('[data-role=status]').textContent='Untrained';draw();}
 const sigmoid=z=>z>=0?1/(1+Math.exp(-z)):Math.exp(z)/(1+Math.exp(z));
 function forward(x,y){const h=model.h,a1=makeArray(h,j=>Math.tanh(x*model.W1[j]+y*model.W1[h+j]+model.b1[j]));const a2=makeArray(h,k=>{let z=model.b2[k];for(let j=0;j<h;j++)z+=a1[j]*model.W2[j*h+k];return Math.tanh(z)});let z=model.b3[0];for(let k=0;k<h;k++)z+=a2[k]*model.W3[k];return{a1,a2,p:sigmoid(z)}}
 function gradients(){const h=model.h,g={W1:makeArray(2*h,()=>0),b1:makeArray(h,()=>0),W2:makeArray(h*h,()=>0),b2:makeArray(h,()=>0),W3:makeArray(h,()=>0),b3:[0]};for(const q of data){const f=forward(q.x,q.y),d3=f.p-q.label,d2=makeArray(h,k=>(1-f.a2[k]**2)*model.W3[k]*d3),d1=makeArray(h,j=>{let s=0;for(let k=0;k<h;k++)s+=model.W2[j*h+k]*d2[k];return(1-f.a1[j]**2)*s});g.b3[0]+=d3;for(let k=0;k<h;k++){g.W3[k]+=f.a2[k]*d3;g.b2[k]+=d2[k];for(let j=0;j<h;j++)g.W2[j*h+k]+=f.a1[j]*d2[k]}for(let j=0;j<h;j++){g.b1[j]+=d1[j];g.W1[j]+=q.x*d1[j];g.W1[h+j]+=q.y*d1[j]}}return g}
 function step(){const g=gradients(),lr=value('rate'),beta1=.9,beta2=.999;model.t++;for(const key of ['W1','b1','W2','b2','W3','b3'])for(let i=0;i<model[key].length;i++){const grad=g[key][i]/data.length;model['m'+key][i]=beta1*model['m'+key][i]+(1-beta1)*grad;model['v'+key][i]=beta2*model['v'+key][i]+(1-beta2)*grad*grad;const mh=model['m'+key][i]/(1-beta1**model.t),vh=model['v'+key][i]/(1-beta2**model.t);model[key][i]-=lr*mh/(Math.sqrt(vh)+1e-8)}}
 async function train(){const token=++runToken,total=value('epochs'),button=root.querySelector('[data-role=train]');button.disabled=true;for(let epoch=0;epoch<total&&token===runToken;epoch++){step();if(epoch%10===0){root.querySelector('[data-role=status]').textContent=`Training ${epoch}/${total}…`;await new Promise(resolve=>setTimeout(resolve,0))}}if(token===runToken){root.querySelector('[data-role=status]').textContent=`Trained for ${total} epochs`;draw()}button.disabled=false}
 function color(p){const lo=[59,76,192],mid=[245,245,245],hi=[180,4,38],a=p<.5?lo:mid,b=p<.5?mid:hi,t=p<.5?p*2:(p-.5)*2;return`rgb(${[0,1,2].map(i=>Math.round((a[i]+(b[i]-a[i])*t)*.48+255*.52)).join(',')})`}
 const sx=x=>area.left+(x+1.15)/2.3*(area.right-area.left),sy=y=>area.bottom-(y+1.15)/2.3*(area.bottom-area.top),ux=p=>(p-area.left)/(area.right-area.left)*2.3-1.15,uy=p=>(area.bottom-p)/(area.bottom-area.top)*2.3-1.15;
 function crossing(a,va,b,vb,t){if((va<t)===(vb<t))return null;const f=(t-va)/(vb-va);return[a[0]+f*(b[0]-a[0]),a[1]+f*(b[1]-a[1])]}
 function draw(){ctx.clearRect(0,0,canvas.width,canvas.height);const cols=80,rows=56,cw=(area.right-area.left)/cols,ch=(area.bottom-area.top)/rows,ps=makeArray(rows+1,r=>makeArray(cols+1,c=>forward(ux(area.left+c*cw),uy(area.top+r*ch)).p));for(let r=0;r<rows;r++)for(let c=0;c<cols;c++){ctx.fillStyle=color((ps[r][c]+ps[r+1][c]+ps[r][c+1]+ps[r+1][c+1])/4);ctx.fillRect(area.left+c*cw,area.top+r*ch,cw+1,ch+1)}ctx.strokeStyle='#d1d5db';ctx.lineWidth=1;ctx.fillStyle='#555';ctx.font='12px Arial';ctx.textAlign='center';for(let i=0;i<=4;i++){const px=area.left+i*(area.right-area.left)/4,py=area.top+i*(area.bottom-area.top)/4;ctx.beginPath();ctx.moveTo(px,area.top);ctx.lineTo(px,area.bottom);ctx.stroke();ctx.beginPath();ctx.moveTo(area.left,py);ctx.lineTo(area.right,py);ctx.stroke();ctx.fillText((-1.15+i*2.3/4).toFixed(1),px,area.bottom+19);ctx.textAlign='right';ctx.fillText((1.15-i*2.3/4).toFixed(1),area.left-8,py+4);ctx.textAlign='center'}ctx.strokeStyle='#333';ctx.lineWidth=1.5;ctx.strokeRect(area.left,area.top,area.right-area.left,area.bottom-area.top);const threshold=value('threshold');ctx.save();ctx.strokeStyle='#111';ctx.lineWidth=2.2;ctx.setLineDash([7,5]);for(let r=0;r<rows;r++)for(let c=0;c<cols;c++){const x0=area.left+c*cw,x1=x0+cw,y0=area.top+r*ch,y1=y0+ch,v=[ps[r][c],ps[r][c+1],ps[r+1][c+1],ps[r+1][c]],hits=[crossing([x0,y0],v[0],[x1,y0],v[1],threshold),crossing([x1,y0],v[1],[x1,y1],v[2],threshold),crossing([x1,y1],v[2],[x0,y1],v[3],threshold),crossing([x0,y1],v[3],[x0,y0],v[0],threshold)].filter(Boolean);for(let i=0;i+1<hits.length;i+=2){ctx.beginPath();ctx.moveTo(...hits[i]);ctx.lineTo(...hits[i+1]);ctx.stroke()}}ctx.restore();for(const q of data){ctx.beginPath();ctx.arc(sx(q.x),sy(q.y),4,0,2*Math.PI);ctx.fillStyle=q.label?'#dc143c':'#4169e1';ctx.fill();ctx.strokeStyle='rgba(255,255,255,.8)';ctx.lineWidth=.8;ctx.stroke()}ctx.fillStyle='#333';ctx.font='15px Arial';ctx.textAlign='center';ctx.fillText('Feature 0',(area.left+area.right)/2,600);ctx.save();ctx.translate(17,(area.top+area.bottom)/2);ctx.rotate(-Math.PI/2);ctx.fillText('Feature 1',0,0);ctx.restore();let loss=0,correct=0;for(const q of data){const p=forward(q.x,q.y).p,clipped=Math.max(1e-10,Math.min(1-1e-10,p));loss-=q.label*Math.log(clipped)+(1-q.label)*Math.log(1-clipped);correct+=((p>=threshold)?1:0)===q.label}root.querySelector('[data-role=loss]').textContent=(loss/data.length).toFixed(4);root.querySelector('[data-role=accuracy]').textContent=`${(100*correct/data.length).toFixed(1)}%`;}
 for(const n of names)knobs[n].addEventListener('input',()=>{showKnobs();if(['samples','noise','turns','seed'].includes(n))makeData();else if(n==='width')resetModel();else if(n==='threshold')draw()});
 root.querySelector('[data-role=train]').addEventListener('click',train);root.querySelector('[data-role=reset]').addEventListener('click',resetModel);root.querySelector('[data-role=new-data]').addEventListener('click',()=>{knobs.seed.value=value('seed')%99+1;showKnobs();makeData()});
 showKnobs();makeData();setTimeout(train,50);
})();
</script>
"""
display(HTML(spiral_widget))

## End.